# Kaggle Diabetes Prediction Dataset - Exploratory Data Analysis
### DiaFlux: Diabetes Risk Prediction and Lifestyle Simulation System

**Author:** DiaFlux Team  
**Date:** March 3, 2026  

---

## Objective
This notebook performs comprehensive exploratory data analysis (EDA) on the **Kaggle Diabetes Prediction Dataset** (`diabetes_prediction_dataset.csv`).

This dataset contains patient health data including lifestyle factors, comorbidities, and demographic information to predict diabetes risk.

---
## Import Required Libraries

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization settings — compatible with all matplotlib versions
available_styles = plt.style.available
if 'seaborn-v0_8-darkgrid' in available_styles:
    plt.style.use('seaborn-v0_8-darkgrid')   # matplotlib >= 3.6
elif 'seaborn-darkgrid' in available_styles:
    plt.style.use('seaborn-darkgrid')         # matplotlib < 3.6
else:
    plt.style.use('ggplot')                   # universal fallback

sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print(f"   pandas     : {pd.__version__}")
print(f"   numpy      : {np.__version__}")
print(f"   matplotlib : {matplotlib.__version__}")
print(f"   seaborn    : {sns.__version__}")---
## 1. Load Dataset

In [ ]:
# Load the Kaggle diabetes prediction dataset
df = pd.read_csv('../data/raw/diabetes_prediction_dataset.csv')

print("📊 Kaggle Diabetes Prediction Dataset Loaded")
print("="*60)

In [ ]:
# Display first 5 rows
print("\n🔍 First 5 Rows:")
df.head()

In [ ]:
# Display dataset shape
print(f"\n📐 Dataset Shape: {df.shape}")
print(f"   • Rows (samples): {df.shape[0]:,}")
print(f"   • Columns (features): {df.shape[1]}")

In [ ]:
# Display column names and types
print("\n📋 Column Names and Types:")
for idx, (col, dtype) in enumerate(zip(df.columns, df.dtypes), 1):
    print(f"   {idx}. {col}: {dtype}")

In [ ]:
# Get detailed info
print("\n📋 Dataset Information:")
df.info()

---
## 2. Basic Information

In [ ]:
# Check for missing values
print("\n❓ Missing Values:")
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Percentage': missing_percentage
})
print(missing_df[missing_df['Missing Count'] > 0])

if missing_values.sum() == 0:
    print("   ✅ No missing values detected!")

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\n🔄 Duplicate Rows: {duplicates:,}")
if duplicates == 0:
    print("   ✅ No duplicate rows found!")
else:
    print(f"   ⚠️ Found {duplicates:,} duplicate rows ({duplicates/len(df)*100:.2f}%)")
    print(f"   💡 Consider removing duplicates before model training")

In [ ]:
# Statistical summary for numerical features
print("\n📊 Statistical Summary (Numerical Features):")
df.describe().T

In [ ]:
# Statistical summary for categorical features
print("\n📊 Statistical Summary (Categorical Features):")
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    df[categorical_cols].describe().T

In [ ]:
# Class distribution of target variable
print("\n🎯 Target Variable Distribution (diabetes):")
target_counts = df['diabetes'].value_counts()
target_percentages = df['diabetes'].value_counts(normalize=True) * 100

target_dist = pd.DataFrame({
    'Count': target_counts,
    'Percentage': target_percentages
})
target_dist.index = ['No Diabetes (0)', 'Diabetes (1)']
print(target_dist)

# Calculate class imbalance ratio
ratio = target_counts[0] / target_counts[1]
print(f"\n   ⚖️ Class Imbalance Ratio: {ratio:.2f}:1 (Non-Diabetic:Diabetic)")
print(f"   ⚠️ Severe class imbalance - requires careful handling!")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Countplot
sns.countplot(x='diabetes', data=df, ax=axes[0], palette='Set3')
axes[0].set_title('Class Distribution - Kaggle Dataset', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Diabetes (0 = No, 1 = Yes)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_xticklabels(['No Diabetes', 'Diabetes'])

# Add value labels on bars
for container in axes[0].containers:
    axes[0].bar_label(container)

# Pie chart
colors = sns.color_palette('Set3')[0:2]
axes[1].pie(target_counts, labels=['No Diabetes', 'Diabetes'], autopct='%1.1f%%', 
            startangle=90, colors=colors, explode=(0, 0.1))
axes[1].set_title('Class Distribution - Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Class distribution visualized")

---
## 3. Numerical Feature Analysis

In [ ]:
# Get numerical columns (excluding target)
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
if 'diabetes' in numerical_features:
    numerical_features.remove('diabetes')

print(f"\n🔢 Numerical Features ({len(numerical_features)}):")
for idx, col in enumerate(numerical_features, 1):
    print(f"   {idx}. {col}")

In [ ]:
# Plot histograms for all numerical features
n_features = len(numerical_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.ravel() if n_features > 1 else [axes]

for idx, col in enumerate(numerical_features):
    axes[idx].hist(df[col], bins=40, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)
    axes[idx].set_ylabel('Frequency', fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

# Remove extra subplots
for idx in range(n_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Kaggle Dataset - Distribution of Numerical Features', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("✅ Histograms plotted")

In [ ]:
# Plot boxplots to detect outliers
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.ravel() if n_features > 1 else [axes]

for idx, col in enumerate(numerical_features):
    sns.boxplot(y=df[col], ax=axes[idx], color='salmon')
    axes[idx].set_title(f'Outlier Detection in {col}', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel(col, fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

# Remove extra subplots
for idx in range(n_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Kaggle Dataset - Outlier Detection (Boxplots)', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("✅ Boxplots for outlier detection plotted")

In [ ]:
# Correlation heatmap (numerical features only)
plt.figure(figsize=(10, 8))
numerical_df = df[numerical_features + ['diabetes']]
correlation_matrix = numerical_df.corr()

sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='RdYlBu_r', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Kaggle Dataset - Correlation Heatmap (Numerical Features)', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✅ Correlation heatmap plotted")

In [ ]:
# Identify highly correlated features (threshold > 0.5)
print("\n🔗 Highly Correlated Feature Pairs (|correlation| > 0.5):")
print("="*60)

# Get upper triangle of correlation matrix
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find features with correlation > 0.5
high_corr = [(column, row, upper_triangle.loc[row, column]) 
             for column in upper_triangle.columns 
             for row in upper_triangle.index 
             if abs(upper_triangle.loc[row, column]) > 0.5]

if high_corr:
    for feat1, feat2, corr in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f"   • {feat1} ↔ {feat2}: {corr:.3f}")
else:
    print("   ✅ No highly correlated feature pairs found (threshold > 0.5)")

---
## 4. Target Variable Analysis

In [ ]:
# Compare numerical features against target variable using boxplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.ravel() if n_features > 1 else [axes]

for idx, col in enumerate(numerical_features):
    sns.boxplot(x='diabetes', y=col, data=df, ax=axes[idx], palette='pastel')
    axes[idx].set_title(f'{col} vs Diabetes Outcome', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Diabetes (0 = No, 1 = Yes)', fontsize=10)
    axes[idx].set_ylabel(col, fontsize=10)
    axes[idx].set_xticklabels(['No Diabetes', 'Diabetes'])
    axes[idx].grid(axis='y', alpha=0.3)

# Remove extra subplots
for idx in range(n_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Kaggle Dataset - Feature Comparison by Diabetes Outcome', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("✅ Feature comparison by target plotted")

In [ ]:
# Compute mean values grouped by target
print("\n📊 Mean Values Grouped by Diabetes Outcome:")
print("="*60)

mean_by_outcome = df[numerical_features + ['diabetes']].groupby('diabetes').mean().T
mean_by_outcome.columns = ['No Diabetes (0)', 'Diabetes (1)']
mean_by_outcome['Difference'] = mean_by_outcome['Diabetes (1)'] - mean_by_outcome['No Diabetes (0)']
mean_by_outcome['% Change'] = (mean_by_outcome['Difference'] / mean_by_outcome['No Diabetes (0)']) * 100

print(mean_by_outcome.sort_values('% Change', ascending=False))

In [ ]:
# Identify strongest predictors based on correlation with target
print("\n🎯 Feature Correlation with Target (diabetes):")
print("="*60)

target_correlation = correlation_matrix['diabetes'].drop('diabetes').sort_values(ascending=False)
print(target_correlation)

# Visualize
plt.figure(figsize=(10, 6))
target_correlation.plot(kind='barh', color='darkgreen', edgecolor='black')
plt.title('Kaggle Dataset - Feature Correlation with Diabetes Outcome', 
          fontsize=14, fontweight='bold')
plt.xlabel('Correlation Coefficient', fontsize=11)
plt.ylabel('Features', fontsize=11)
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Top 3 strongest predictors:")
for idx, (feature, corr) in enumerate(target_correlation.head(3).items(), 1):
    print(f"   {idx}. {feature}: {corr:.3f}")

---
## 5. Categorical Feature Analysis

In [ ]:
# Get categorical columns
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print(f"\n📊 Categorical Features ({len(categorical_features)}):")
for idx, col in enumerate(categorical_features, 1):
    print(f"   {idx}. {col}")
    print(f"      Unique values: {df[col].nunique()}")
    print(f"      Categories: {df[col].unique()[:10]}")
    print()

In [ ]:
# Analyze Gender distribution
if 'gender' in df.columns:
    print("\n👥 Gender Distribution:")
    print("="*60)
    gender_dist = df['gender'].value_counts()
    gender_pct = df['gender'].value_counts(normalize=True) * 100
    
    gender_df = pd.DataFrame({
        'Count': gender_dist,
        'Percentage': gender_pct
    })
    print(gender_df)
    
    # Plot gender distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Overall gender distribution
    sns.countplot(x='gender', data=df, ax=axes[0], palette='viridis')
    axes[0].set_title('Gender Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Gender', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    for container in axes[0].containers:
        axes[0].bar_label(container)
    
    # Gender vs Diabetes
    sns.countplot(x='gender', hue='diabetes', data=df, ax=axes[1], palette='Set2')
    axes[1].set_title('Gender vs Diabetes Outcome', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Gender', fontsize=11)
    axes[1].set_ylabel('Count', fontsize=11)
    axes[1].legend(title='Diabetes', labels=['No', 'Yes'])
    
    plt.tight_layout()
    plt.show()
    
    # Diabetes rate by gender
    print("\n📊 Diabetes Rate by Gender:")
    diabetes_by_gender = df.groupby('gender')['diabetes'].agg(['sum', 'count', 'mean'])
    diabetes_by_gender.columns = ['Diabetic Count', 'Total Count', 'Diabetes Rate']
    diabetes_by_gender['Diabetes Rate'] = diabetes_by_gender['Diabetes Rate'] * 100
    print(diabetes_by_gender)

In [ ]:
# Analyze Smoking History
if 'smoking_history' in df.columns:
    print("\n🚬 Smoking History Distribution:")
    print("="*60)
    smoking_dist = df['smoking_history'].value_counts()
    smoking_pct = df['smoking_history'].value_counts(normalize=True) * 100
    
    smoking_df = pd.DataFrame({
        'Count': smoking_dist,
        'Percentage': smoking_pct
    })
    print(smoking_df)
    
    # Plot smoking distribution
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Overall smoking distribution
    sns.countplot(y='smoking_history', data=df, ax=axes[0], 
                  order=smoking_dist.index, palette='rocket')
    axes[0].set_title('Smoking History Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Count', fontsize=11)
    axes[0].set_ylabel('Smoking History', fontsize=11)
    
    # Smoking vs Diabetes
    sns.countplot(y='smoking_history', hue='diabetes', data=df, ax=axes[1],
                  order=smoking_dist.index, palette='Set1')
    axes[1].set_title('Smoking History vs Diabetes Outcome', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Count', fontsize=11)
    axes[1].set_ylabel('Smoking History', fontsize=11)
    axes[1].legend(title='Diabetes', labels=['No', 'Yes'])
    
    plt.tight_layout()
    plt.show()
    
    # Diabetes rate by smoking history
    print("\n📊 Diabetes Rate by Smoking History:")
    diabetes_by_smoking = df.groupby('smoking_history')['diabetes'].agg(['sum', 'count', 'mean'])
    diabetes_by_smoking.columns = ['Diabetic Count', 'Total Count', 'Diabetes Rate']
    diabetes_by_smoking['Diabetes Rate'] = diabetes_by_smoking['Diabetes Rate'] * 100
    print(diabetes_by_smoking.sort_values('Diabetes Rate', ascending=False))

In [ ]:
# Analyze Hypertension
if 'hypertension' in df.columns:
    print("\n💓 Hypertension Distribution:")
    print("="*60)
    hypertension_dist = df['hypertension'].value_counts()
    hypertension_pct = df['hypertension'].value_counts(normalize=True) * 100
    
    hypertension_df = pd.DataFrame({
        'Count': hypertension_dist,
        'Percentage': hypertension_pct
    })
    hypertension_df.index = ['No Hypertension (0)', 'Hypertension (1)']
    print(hypertension_df)
    
    # Plot hypertension vs diabetes
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Hypertension distribution
    sns.countplot(x='hypertension', data=df, ax=axes[0], palette='magma')
    axes[0].set_title('Hypertension Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Hypertension (0 = No, 1 = Yes)', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    axes[0].set_xticklabels(['No', 'Yes'])
    for container in axes[0].containers:
        axes[0].bar_label(container)
    
    # Hypertension vs Diabetes
    sns.countplot(x='hypertension', hue='diabetes', data=df, ax=axes[1], palette='cool')
    axes[1].set_title('Hypertension vs Diabetes Outcome', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Hypertension (0 = No, 1 = Yes)', fontsize=11)
    axes[1].set_ylabel('Count', fontsize=11)
    axes[1].set_xticklabels(['No', 'Yes'])
    axes[1].legend(title='Diabetes', labels=['No', 'Yes'])
    
    plt.tight_layout()
    plt.show()
    
    # Diabetes rate by hypertension
    print("\n📊 Diabetes Rate by Hypertension:")
    diabetes_by_hypertension = df.groupby('hypertension')['diabetes'].agg(['sum', 'count', 'mean'])
    diabetes_by_hypertension.columns = ['Diabetic Count', 'Total Count', 'Diabetes Rate']
    diabetes_by_hypertension['Diabetes Rate'] = diabetes_by_hypertension['Diabetes Rate'] * 100
    diabetes_by_hypertension.index = ['No Hypertension', 'Hypertension']
    print(diabetes_by_hypertension)

In [ ]:
# Analyze Heart Disease
if 'heart_disease' in df.columns:
    print("\n❤️ Heart Disease Distribution:")
    print("="*60)
    heart_dist = df['heart_disease'].value_counts()
    heart_pct = df['heart_disease'].value_counts(normalize=True) * 100
    
    heart_df = pd.DataFrame({
        'Count': heart_dist,
        'Percentage': heart_pct
    })
    heart_df.index = ['No Heart Disease (0)', 'Heart Disease (1)']
    print(heart_df)
    
    # Plot heart disease vs diabetes
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Heart disease distribution
    sns.countplot(x='heart_disease', data=df, ax=axes[0], palette='Reds')
    axes[0].set_title('Heart Disease Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Heart Disease (0 = No, 1 = Yes)', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    axes[0].set_xticklabels(['No', 'Yes'])
    for container in axes[0].containers:
        axes[0].bar_label(container)
    
    # Heart disease vs Diabetes
    sns.countplot(x='heart_disease', hue='diabetes', data=df, ax=axes[1], palette='RdYlGn_r')
    axes[1].set_title('Heart Disease vs Diabetes Outcome', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Heart Disease (0 = No, 1 = Yes)', fontsize=11)
    axes[1].set_ylabel('Count', fontsize=11)
    axes[1].set_xticklabels(['No', 'Yes'])
    axes[1].legend(title='Diabetes', labels=['No', 'Yes'])
    
    plt.tight_layout()
    plt.show()
    
    # Diabetes rate by heart disease
    print("\n📊 Diabetes Rate by Heart Disease:")
    diabetes_by_heart = df.groupby('heart_disease')['diabetes'].agg(['sum', 'count', 'mean'])
    diabetes_by_heart.columns = ['Diabetic Count', 'Total Count', 'Diabetes Rate']
    diabetes_by_heart['Diabetes Rate'] = diabetes_by_heart['Diabetes Rate'] * 100
    diabetes_by_heart.index = ['No Heart Disease', 'Heart Disease']
    print(diabetes_by_heart)

---
## 6. Key Insights and Findings

### **1. Dataset Characteristics:**
- Large dataset with 100,000 samples
- Mix of numerical and categorical features
- Comprehensive health profile including lifestyle factors
- Suitable for robust machine learning model training

### **2. Class Imbalance:**
- Severe class imbalance (>10:1 ratio)
- Requires aggressive resampling techniques (SMOTE, ADASYN)
- Class weights essential for model training
- Stratified sampling crucial for validation

### **3. Key Risk Factors (Numerical):**
- **HbA1c level**: Strongest predictor of diabetes
- **Blood glucose level**: Very strong correlation
- **BMI**: Positive correlation with diabetes risk
- **Age**: Moderate positive correlation

### **4. Categorical Risk Factors:**
- **Hypertension**: Significantly higher diabetes rates in hypertensive patients
- **Heart Disease**: Strong association with diabetes
- **Smoking History**: Varying diabetes rates across categories
- **Gender**: Potential differences in diabetes prevalence

### **5. Data Quality:**
- No missing values (clean dataset)
- Contains duplicate rows that need removal
- No obvious zero-value issues (unlike Pima dataset)
- Real-world data with diverse patient profiles

### **6. Lifestyle Factors for DiaFlux:**
- **Smoking history** enables lifestyle simulation scenarios
- **BMI** can be modified through diet/exercise simulations
- **Comorbidities** (hypertension, heart disease) show intervention importance
- Perfect for "what-if" analysis in lifestyle simulation component

---

## 7. Preprocessing Recommendations

### **Required Steps:**

1. **Remove Duplicates:**
   - Drop duplicate rows before training
   - Verify if duplicates are true duplicates or data entry errors

2. **Encode Categorical Variables:**
   - One-hot encoding for gender
   - Ordinal encoding or one-hot for smoking_history
   - Binary encoding already present for hypertension/heart_disease

3. **Handle Class Imbalance:**
   - Apply SMOTE or ADASYN for synthetic minority samples
   - Use class weights in model training
   - Consider ensemble methods (XGBoost, LightGBM)
   - Stratified sampling for train/validation/test split

4. **Feature Scaling:**
   - Standardization (StandardScaler) for tree-based models
   - Normalization (MinMaxScaler) for neural networks
   - Robust scaling if outliers present

5. **Feature Engineering:**
   - Age groups (young, middle-aged, senior)
   - BMI categories (underweight, normal, overweight, obese)
   - Interaction features (hypertension × heart_disease)
   - Risk scores combining multiple factors
   - HbA1c categories (normal, prediabetic, diabetic)

6. **Model Selection:**
   - Logistic Regression (baseline)
   - Random Forest (handles imbalance well)
   - XGBoost/LightGBM (excellent for tabular data)
   - Neural Networks (given large dataset size)

7. **Validation Strategy:**
   - Stratified k-fold cross-validation (k=5 or 10)
   - Monitor ROC-AUC, Precision-Recall
   - Focus on recall for diabetic class (minimize false negatives)

8. **Explainability:**
   - SHAP values for feature importance
   - LIME for individual predictions
   - Important for healthcare applications

---

---
## 📝 Conclusion

The Kaggle Diabetes Prediction Dataset is **ideal for DiaFlux**:

✅ **Strengths:**
- Large sample size (100,000 records)
- Rich feature set including lifestyle factors
- Perfect for lifestyle simulation component
- Includes important comorbidities
- Clean data with no missing values
- Real-world applicability

⚠️ **Challenges:**
- Severe class imbalance requires careful handling
- Contains duplicate rows
- Categorical features need encoding
- Larger computational requirements

🎯 **Perfect for DiaFlux Because:**
1. **Risk Prediction**: Strong predictors (HbA1c, glucose, BMI)
2. **Lifestyle Simulation**: Smoking, BMI, comorbidities enable "what-if" scenarios
3. **Robust Models**: Large dataset supports complex models
4. **Actionable Insights**: Features map to modifiable risk factors

**Recommended as PRIMARY dataset for DiaFlux project!**

**Next Steps:**
1. Implement data preprocessing pipeline
2. Handle duplicates and encode categorical features
3. Apply SMOTE/ADASYN for class imbalance
4. Build and compare multiple models
5. Develop lifestyle simulation features
6. Implement explainable AI components

---

*End of Kaggle Dataset EDA Notebook*